# Etapa 2 - entrenamiento y comparacion de modelos

Notebook pensada para Colab GPU o para replicar localmente los experimentos finales del TP. El objetivo fue comparar un **ResNet18 fine-tuned** contra una **CNN custom** y luego reutilizar el mejor modelo en las Etapas 1 y 3.

## Resultado final

- **ResNet18 fine-tuned**: accuracy test `0.9429`, precision `0.9486`, recall `0.9429`, specificity `0.9992`, F1 `0.9432`.
- **CNN custom**: accuracy test `0.5043`, precision `0.5177`, recall `0.5043`, specificity `0.9928`, F1 `0.4800`.
- **Etapa 1 / similitud** sobre `valid` (3 imagenes por raza, 210 consultas, top-k=10): baseline accuracy `0.9429` NDCG@10 `0.9460`, resnet accuracy `0.9476` NDCG@10 `0.9429`, cnn accuracy `0.5190` NDCG@10 `0.6572`.

Conclusion: **ResNet18 fine-tuned** queda como modelo final recomendado tanto para clasificacion supervisada como para embeddings de similitud.

In [ ]:
# Si corres esto en Colab
!git clone <https://github.com/Lauaf/TP2_VC>
%cd TP2_VC
!pip install -r requirements.txt
!python --version

In [ ]:
# Dataset (si usas Kaggle/Colab)
import kagglehub
path = kagglehub.dataset_download("gpiosenka/70-dog-breedsimage-data-set")
print(path)

In [ ]:
# Paths utiles
from pathlib import Path
ROOT = Path.cwd()
DATASET = ROOT / "data" / "dataset"
MODELS = ROOT / "models"
OUTPUT = ROOT / "output"
DATASET, MODELS, OUTPUT

## Preprocesamiento y augmentation

- Resize a `224x224`.
- Normalizacion ImageNet.
- En entrenamiento: `RandomHorizontalFlip`, `RandomRotation(12)` y `ColorJitter`.
- Se mantuvieron los splits originales `train / valid / test` del dataset.

In [ ]:
# Entrenamiento recomendado en GPU
import os
os.environ["TRAIN_EPOCHS"] = "6"
os.environ["TRAIN_BATCH_SIZE"] = "64"
os.environ["TRAIN_NUM_WORKERS"] = "6"
os.environ["TRAIN_LR"] = "0.0003"
!python scripts/train_classifier.py --model resnet18_finetuned

In [ ]:
# CNN custom para comparacion
import os
os.environ["TRAIN_EPOCHS"] = "40"
os.environ["TRAIN_BATCH_SIZE"] = "64"
os.environ["TRAIN_NUM_WORKERS"] = "6"
os.environ["TRAIN_LR"] = "0.0003"
!python scripts/train_classifier.py --model cnn_custom

## Evidencia visual

### Curvas de entrenamiento
![Training curves](report_assets/classifier_history.png)

### Comparacion de metricas de clasificacion
![Classifier metrics](report_assets/classifier_metrics.png)

### Comparacion de similitud
![Similarity metrics](report_assets/similarity_metrics.png)

### Matrices de confusion
![ResNet confusion](report_assets/resnet_confusion_matrix.png)

![CNN confusion](report_assets/cnn_confusion_matrix.png)

## Analisis

- El **ResNet18 fine-tuned** supera ampliamente a la CNN custom en clasificacion y tambien produce mejores embeddings para recuperacion por similitud.
- La CNN custom logra aprender parcialmente algunas razas frecuentes, pero generaliza mal en clases visualmente parecidas.
- En las pruebas manuales del pipeline final, una imagen `Beagle/01.jpg` fue reconocida correctamente en las tres etapas con ResNet: similitud, clasificacion y deteccion + clasificacion.

## Artefactos esperados

- `models/resnet18_finetuned.pth`
- `models/cnn_custom.pth`
- `output/resnet18_finetuned_history.json`
- `output/resnet18_finetuned_metrics.json`
- `output/cnn_custom_history.json`
- `output/cnn_custom_metrics.json`
- `report_assets/*.png`